# 🧬 Kaggle 示範：細胞追蹤入門 Baseline（最近鄰連線法）

**比賽**：[Biohub - Cell Tracking During Development](https://www.kaggle.com/competitions/biohub-cell-tracking-during-development)

這份 notebook 是給課堂示範用的「起手式」：帶你走完一個 Kaggle 競賽從**讀資料 → 偵測 → 追蹤 → 產生 `submission.csv`** 的完整一條龍。跑完大約 **50 秒**，會拿到 **約 0.14 分**——分數不高，但**重點是先完整提交一次**，把流程跑通，之後再慢慢加強。

> 🙏 **致謝與授權**：演算法構想改編自官方起手式 notebook *"Cell Tracking Getting Started w/ Nearest Neighbor"*（Kaggle 帳號 inversion，Apache-2.0 授權）。本檔為教學重寫版，加上中文逐步解說。

---
### 這支 baseline 的四步驟

1. **讀影像**：每筆樣本是一段 3D+時間的顯微影片（`.zarr`），形狀約 `(時間 T, 深度 Z, 高 Y, 寬 X) = (100, 64, 256, 256)`。
2. **偵測細胞**：每個時間點 → 平滑 → 取亮度前 10% 當門檻二值化 → 連通元件 → 每團當一顆細胞，記下它的質心座標（這就是一個 `node`）。
3. **跨時間連線**：相鄰兩個時間點之間，用「最近鄰 + 匈牙利演算法」把同一顆細胞配對起來（這就是一條 `edge`）。
4. **輸出**：把所有 node 與 edge 寫成 `submission.csv`。

## 0. 先認識資料與評分（看懂規則再動手）

**資料格式**
- 影像存成 `.zarr` 資料夾，主陣列在 `0/`，形狀 `(T, Z, Y, X)`、`uint16`。
- 每個時間點是一個壓縮 chunk，路徑 `0/c/{t}/0/0/0`，用 blosc/zstd 壓縮。
- 中繼資料（形狀、型別）在 `0/zarr.json`。
- 體素的真實尺度：`z=1.625`、`y=x=0.40625` µm/voxel。

**提交格式**：一個 CSV，含 `node`（細胞偵測）與 `edge`（連線）兩種列，欄位為：
```
id,dataset,row_type,node_id,t,z,y,x,source_id,target_id
```
- `node` 列：填 `node_id, t, z, y, x`（整數體素座標），`source_id/target_id` 填 `-1`。
- `edge` 列：填 `source_id, target_id`（指向 node_id），其餘填 `-1`。
- `dataset` 欄要等於測試資料夾名稱（去掉 `.zarr`）。**每個測試樣本都要出現在提交裡**。

**評分**：`score = adjusted_edge_jaccard + 0.1 × division_jaccard`（連線正確度為主、加一點細胞分裂偵測）。

In [ ]:
# === 套件 ===
# blosc2：手動解壓 zarr 的 chunk（比賽 notebook 關閉網路，所以不靠 zarr 套件、直接讀檔解壓最穩）
import json
import os

import blosc2
import numpy as np
import pandas as pd
from scipy.ndimage import label, uniform_filter
from scipy.optimize import linear_sum_assignment

print('套件載入完成')

In [ ]:
# === 參數設定（先用官方 baseline 的數字，之後可自己調）===
TEST_DIR = '/kaggle/input/competitions/biohub-cell-tracking-during-development/test'

SCALE = np.array([1.625, 0.40625, 0.40625])  # (Z, Y, X) µm/voxel：算真實距離用
DOWNSAMPLE = 4         # 影像降採樣倍率：跑快一點（256→64）
PERCENTILE = 90        # 亮度門檻：取最亮的前 10% 當「可能是細胞」
MAX_LINK_DISTANCE = 15.0  # µm：兩個時間點的細胞距離超過這個就不連線

# 註：若路徑找不到資料，請到右側 + Add Input 把本比賽資料掛上，
#     並確認資料夾名稱與上面 TEST_DIR 一致。
print('TEST_DIR =', TEST_DIR)

## 1. 讀一張影像：把一個時間點的 3D 體積讀進來

先寫兩個小工具：一個讀 `.zarr` 的中繼資料（形狀、型別），一個讀指定時間點的 3D 體積。

> 💡 教學點：`.zarr` 不是一個檔案，而是一個**資料夾**。每個時間點被切成一塊壓縮的 chunk 分開存。我們用 `blosc2.decompress` 解壓，再用 `np.frombuffer` 還原成陣列。

In [ ]:
def read_zarr_meta(zarr_path):
    """讀 0/zarr.json，回傳 (shape, dtype)。shape = (T, Z, Y, X)。"""
    with open(os.path.join(zarr_path, '0', 'zarr.json')) as f:
        meta = json.load(f)
    shape = tuple(meta['shape'])
    dtype = np.dtype(meta['data_type'])  # 例如 'uint16'
    return shape, dtype


def read_timepoint(zarr_path, t, shape, dtype):
    """讀第 t 個時間點的 3D 體積，回傳 (Z, Y, X) 陣列；chunk 不存在（全黑）就回傳全 0。"""
    chunk_path = os.path.join(zarr_path, '0', 'c', str(t), '0', '0', '0')
    if not os.path.exists(chunk_path):
        return np.zeros(shape[1:], dtype=dtype)  # zarr 不會存全 0 的 chunk
    with open(chunk_path, 'rb') as f:
        compressed = f.read()
    decompressed = blosc2.decompress(compressed)
    return np.frombuffer(decompressed, dtype=dtype).reshape(shape[1:])  # (Z, Y, X)


print('工具函式定義完成')

### （選看）瞄一眼資料：把某個時間點壓平來看

把 3D 體積沿 Z 軸取最大值投影（max projection），就能用一張 2D 圖大致看到細胞分布。**這一格純粹是幫忙理解資料，不影響提交**，看完可以略過。

In [ ]:
import matplotlib.pyplot as plt

sample_folders = sorted(d for d in os.listdir(TEST_DIR) if d.endswith('.zarr'))
print('測試樣本數：', len(sample_folders))

demo_path = os.path.join(TEST_DIR, sample_folders[0])
shape, dtype = read_zarr_meta(demo_path)
print('形狀 (T,Z,Y,X) =', shape, '型別 =', dtype)

vol0 = read_timepoint(demo_path, 0, shape, dtype)       # 第 0 個時間點
plt.figure(figsize=(5, 5))
plt.imshow(vol0.max(axis=0), cmap='magma')              # 沿 Z 軸壓平
plt.title(f'{sample_folders[0]}  t=0  (Z-max projection)')
plt.axis('off')
plt.show()

## 2. 偵測細胞：把一個時間點變成一串質心座標

做法很單純（這就是「baseline」的精神——先求有、再求好）：
1. **降採樣**讓計算量變小。
2. **平滑**（`uniform_filter`）壓掉雜訊。
3. 取亮度的 **90 百分位當門檻**做二值化——比門檻亮的當「細胞候選」。
4. **連通元件**（`label`）把相連的亮點分群，每群算一個**質心**＝一顆細胞。

> ⚠️ 這個方法很粗：細胞貼太近會被當成一顆、暗一點的會漏掉。這正是後面可以改進的地方。

In [ ]:
def detect_cells(vol):
    """輸入一個 (Z,Y,X) 體積，回傳每顆細胞的質心座標 list（原始體素座標 (z,y,x)）。"""
    ds = vol[::DOWNSAMPLE, ::DOWNSAMPLE, ::DOWNSAMPLE]      # 降採樣
    smoothed = uniform_filter(ds.astype(np.float32), size=3)  # 平滑去雜訊
    threshold = np.percentile(smoothed, PERCENTILE)        # 前 10% 亮度當門檻
    binary = smoothed > threshold

    labeled, n = label(binary)                             # 連通元件
    centroids = []
    for i in range(1, n + 1):
        coords = np.argwhere(labeled == i)                 # 這團的所有體素
        c = coords.mean(axis=0) * DOWNSAMPLE               # 質心，乘回原始解析度
        centroids.append(c)                                # (z, y, x)
    return centroids


# 試跑一個時間點看看抓到幾顆
cells0 = detect_cells(vol0)
print(f't=0 偵測到 {len(cells0)} 顆細胞；前 3 顆質心 (z,y,x)：')
for c in cells0[:3]:
    print('  ', np.round(c, 1))

## 3. 跨時間連線：把同一顆細胞在相鄰時間點接起來

已知 `t` 和 `t+1` 各自的細胞質心，怎麼知道誰是誰？
- 算出兩邊**每對細胞之間的真實距離**（µm，用 `SCALE` 把體素換成微米）。
- 用**匈牙利演算法**（`linear_sum_assignment`）找一組「總距離最小」的一對一配對。
- 配對距離若超過 `MAX_LINK_DISTANCE`（15 µm）就丟掉——太遠的不太可能是同一顆。

每留下一組配對，就產生一條 `edge`（從 `t` 的 node 指到 `t+1` 的 node）。

In [ ]:
def link_frames(prev_ids, prev_coords, curr_ids, curr_coords):
    """回傳 [(source_node_id, target_node_id), ...]。coords 為 (z,y,x) 體素座標陣列。"""
    if len(prev_ids) == 0 or len(curr_ids) == 0:
        return []
    # 距離矩陣：先換成微米再算歐氏距離
    diff = prev_coords[:, None, :] - curr_coords[None, :, :]   # (P, C, 3)
    dist = np.sqrt(((diff * SCALE) ** 2).sum(axis=2))          # (P, C)，單位 µm

    row_ind, col_ind = linear_sum_assignment(dist)            # 最小總距離配對
    edges = []
    for r, c in zip(row_ind, col_ind):
        if dist[r, c] <= MAX_LINK_DISTANCE:                   # 太遠就不連
            edges.append((prev_ids[r], curr_ids[c]))
    return edges


print('連線函式定義完成')

## 4. 跑完整測試集，產生 `submission.csv`

把上面三步串起來，對**每一個測試樣本**跑一遍，邊跑邊把 node 列與 edge 列收進 `all_rows`，最後寫成 CSV。

> 提醒：`node_id` 在**每個 dataset 內**從 1 開始重新編號，edge 用的就是同一個 dataset 內的編號。

In [ ]:
test_folder_names = sorted(
    d.replace('.zarr', '') for d in os.listdir(TEST_DIR) if d.endswith('.zarr')
)

all_rows = []
for folder_name in test_folder_names:
    zarr_path = os.path.join(TEST_DIR, folder_name + '.zarr')
    shape, dtype = read_zarr_meta(zarr_path)
    n_t = shape[0]

    node_id = 1                 # 每個 dataset 內重新從 1 編號
    prev_ids, prev_coords = [], np.empty((0, 3))

    for t in range(n_t):
        vol = read_timepoint(zarr_path, t, shape, dtype)
        centroids = detect_cells(vol)

        # —— 收 node 列，並記住這個時間點每顆細胞的 node_id 與座標 ——
        curr_ids, curr_coords = [], []
        for c in centroids:
            z, y, x = int(round(c[0])), int(round(c[1])), int(round(c[2]))
            all_rows.append({
                'dataset': folder_name, 'row_type': 'node', 'node_id': node_id,
                't': t, 'z': z, 'y': y, 'x': x, 'source_id': -1, 'target_id': -1,
            })
            curr_ids.append(node_id)
            curr_coords.append(c)
            node_id += 1
        curr_coords = np.array(curr_coords) if curr_coords else np.empty((0, 3))

        # —— 跟上一個時間點連線，收 edge 列 ——
        for s, d in link_frames(prev_ids, prev_coords, curr_ids, curr_coords):
            all_rows.append({
                'dataset': folder_name, 'row_type': 'edge', 'node_id': -1,
                't': -1, 'z': -1, 'y': -1, 'x': -1, 'source_id': s, 'target_id': d,
            })

        prev_ids, prev_coords = curr_ids, curr_coords

    print(f'{folder_name}: {node_id - 1} 顆細胞')

print('\n全部跑完，總列數：', len(all_rows))

In [ ]:
# === 寫出 submission.csv（檔名一定要叫這個）===
submission = pd.DataFrame(all_rows)
submission.insert(0, 'id', range(len(submission)))          # 第一欄是流水號
submission = submission[['id', 'dataset', 'row_type', 'node_id',
                         't', 'z', 'y', 'x', 'source_id', 'target_id']]
submission.to_csv('submission.csv', index=False)

print('已寫出 submission.csv，前幾列：')
submission.head()

## 5. 提交，然後想怎麼變強

**怎麼提交**
1. 右上角 **Save Version → Save & Run All (Commit)**，讓 notebook 從頭跑一遍。
2. 跑完後，到這支 notebook 的 **Output** 分頁，按 **Submit** 把 `submission.csv` 送出。
3. 到比賽的 **Leaderboard** 看自己的分數。🎉 你已經完整參加了一個 Kaggle 競賽！

> 注意：本比賽是 **Code 競賽**，提交 notebook 必須**關閉網路**、執行時間 ≤ 12 小時、輸出檔名為 `submission.csv`。

**這個 baseline 弱在哪、可以怎麼加強**
- **偵測**：固定門檻太粗 → 試 `peak_local_max`、分水嶺（watershed）、或深度學習分割（如 Cellpose、StarDist）。
- **連線**：只看最近距離 → 加上**運動預測**（卡爾曼濾波）、處理**斷掉再接回**（gap closing）。
- **細胞分裂**：目前完全沒處理 → 偵測一顆變兩顆的 division，能多拿那 `0.1 ×` 的分數。

**想看更高分的公開作法**（同一個比賽的 Code 分頁）：
- 「Biohub Cell Tracking: Data Model, EDA, Baseline」(~0.687，含資料探索)
- 「BioHub Cell Tracking: Metric-Aware Baseline」(~0.637)
- 「Cell TrackingV2: Sub-Voxel Refinement & EdgePruning」(~0.618)

一步一步來：**先提交一次拿到分數，再挑一個地方改進**——這就是參加 Kaggle 競賽最實在的成長方式。